# Shtemaran 292 Benchmark — All Models + Gemini 2.5 Flash Zero-Shot

**Purpose:** Evaluation of BiLSTM, HyeBERT, mBERT, Ensemble, and **zero-shot Gemini 2.5 Flash (CoT)**
on the **Shtemaran gold-standard** participle punctuation sentences.

**Dataset:** 292 sentence pairs from the Armenian grammar reference (*Shtemaran*/*Շտեմարան*):
- `Partial_NP_Final.txt` — sentences with participle punctuation stripped (indexed as `N-M. sentence`)
- `ONLY_Participle-related_punct_in-place.txt` — linguist-corrected versions (same indexing)

**Workflow:**
1. Steps 1-3: Build JSONL (strip indexes, pair sentences)
2. Run `gemini_shtemaran_runner.py` in a separate terminal
3. Steps 4-6: Load models + Gemini cache
4. Steps 7-15: Full evaluation, comparison, plots

## Step 1: Setup & Configuration

In [180]:
import os, re, json, math, time, difflib, random
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (classification_report, f1_score,
    precision_recall_fscore_support, confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HyeBERT available: True
Gemini cache exists: False
All core paths verified ✓


In [ ]:
# Paths
SHTEMARAN_NP     = Path("./Data Full/Partial_NP_Final.txt")
SHTEMARAN_CORR   = Path("./Data Full/ONLY_Participle-related_punct_in-place.txt")
BILSTM_VOCAB     = Path("./Weights - train_bilstm_v3/armenian_vocab.json")
BILSTM_CKPT      = Path("./Weights - train_bilstm_v3/bilstm_v3_best.pt")
HYEBERT_WEIGHTS  = Path("./hyebert_v3_output/hyebert_v3_best.pt")
HYEBERT_NAME     = "aking11/hyebert"
MBERT_WEIGHTS    = Path("./mBert/mbert_best.pt")
MBERT_NAME       = "bert-base-multilingual-cased"
GEMINI_CACHE     = Path("./Data Full/shtemaran_gemini_results.json")

LABEL_MAP  = {"O": 0, "COMMA_AFTER": 1, "BUTH_AFTER": 2, "REMOVE_COMMA": 3}
LABEL_LIST = ["O", "COMMA_AFTER", "BUTH_AFTER", "REMOVE_COMMA"]
NUM_LABELS = 4
ENSEMBLE_ALPHA = 0.45

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")

for p in [SHTEMARAN_NP, SHTEMARAN_CORR, BILSTM_CKPT, MBERT_WEIGHTS]:
    assert p.exists(), f"Missing: {p}"
HYEBERT_AVAILABLE = HYEBERT_WEIGHTS.exists()
print(f"HyeBERT available: {HYEBERT_AVAILABLE}")
print(f"Gemini cache exists: {GEMINI_CACHE.exists()}")
print("All core paths verified \u2713")

## Step 2: Label Pipeline

In [183]:
ARMENIAN_PUNCT = '\u055b\u055c\u055d\u055e\u055f\u0589' # ՛, ՜, ՝, ՞, etc.
ARMENIAN_PUNCT_SET = set(ARMENIAN_PUNCT)

def tokenize_armenian(text):
    if not isinstance(text, str) or not text.strip(): return []
    for ch in ARMENIAN_PUNCT:
        text = text.replace(ch, f' {ch} ')
    return re.findall(r'[\w\u0530-\u058F]+|[^\w\s]', text)

def generate_labels(orig_tokens, corr_tokens):
    labels = [0] * len(orig_tokens)
    sm = difflib.SequenceMatcher(None, orig_tokens, corr_tokens)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal': continue
        if tag in ('delete', 'replace'):
            for i in range(i1, i2):
                if orig_tokens[i] == ',': labels[i] = 3
        if tag in ('insert', 'replace'):
            for j in range(j1, j2):
                tok = corr_tokens[j]
                if tok == ',' and i1 > 0: labels[i1 - 1] = 1
                elif tok == '\u055d' and i1 > 0: labels[i1 - 1] = 2
    return labels

def is_punct_token(tok):
    if len(tok) == 1 and tok in ARMENIAN_PUNCT_SET: return True
    return not re.match(r'[\w\u0530-\u058F]', tok)

def extract_word_labels(orig_tokens, label_ids):
    words, word_labels = [], []
    for tok, lbl in zip(orig_tokens, label_ids):
        if is_punct_token(tok):
            if lbl == 3 and word_labels and word_labels[-1] == 0:
                word_labels[-1] = 3
            continue
        words.append(tok); word_labels.append(lbl)
    return words, word_labels

def reconstruct_sentence(words, labels):
    parts = []
    for w, lbl in zip(words, labels):
        parts.append(w)
        if lbl == 1: parts.append(',')
        elif lbl == 2: parts.append('\u055d')
    return ' '.join(parts)

print("Label pipeline ready \u2713")

Label pipeline ready ✓


## Step 3: Load Shtemaran Data & Build JSONL

Each line in the txt files has the format: `N-M. sentence text`
where N = ordinal (1-292) and M = source text index from the Shtemaran book.
We strip the prefix and use N as the sentence ID.

In [234]:
INDEX_PATTERN = re.compile(r'^(\d+)-(\d+)\.\s*')

def parse_shtemaran_line(line):
    """Parse 'N-M. sentence' format. Returns (ordinal, text_index, sentence)."""
    line = line.strip()
    m = INDEX_PATTERN.match(line)
    if m:
        ordinal = int(m.group(1))   # 1-292 ordinal number
        text_idx = int(m.group(2))  # source text index in Shtemaran book
        sentence = line[m.end():].strip()
        return ordinal, text_idx, sentence
    # Fallback: no index found, use line as-is
    return None, None, line

with open(SHTEMARAN_NP, 'r', encoding='utf-8') as f:
    np_raw = [line.strip() for line in f if line.strip()]
with open(SHTEMARAN_CORR, 'r', encoding='utf-8') as f:
    corr_raw = [line.strip() for line in f if line.strip()]

print(f"Partial_NP_Final.txt: {len(np_raw)} lines")
print(f"ONLY_Participle-related: {len(corr_raw)} lines")
assert len(np_raw) == len(corr_raw), f"Mismatch: {len(np_raw)} vs {len(corr_raw)}"

shtemaran_samples = []
skipped = 0
for raw_np, raw_corr in zip(np_raw, corr_raw):
    ordinal_np, text_idx_np, orig_text = parse_shtemaran_line(raw_np)
    ordinal_corr, text_idx_corr, corr_text = parse_shtemaran_line(raw_corr)

    # Using ordinal from NP file as ID (1-based)
    sentence_id = ordinal_np if ordinal_np is not None else len(shtemaran_samples) + 1

    if not orig_text:
        skipped += 1; continue

    orig_tokens = tokenize_armenian(orig_text)
    corr_tokens = tokenize_armenian(corr_text)
    if not orig_tokens:
        skipped += 1; continue
    if not corr_tokens:
        corr_tokens = orig_tokens

    labels = generate_labels(orig_tokens, corr_tokens)
    words, word_labels = extract_word_labels(orig_tokens, labels)
    if not words:
        skipped += 1; continue

    shtemaran_samples.append({"id": sentence_id,
                        "text_index": text_idx_np,
                        "original": orig_text,
                        "linguist_corrected": corr_text,
                        "words": words,
                        "labels": word_labels})

print(f"\nShtemaran dataset: {len(shtemaran_samples)} sentence pairs")
print(f"  Skipped: {skipped}")
if shtemaran_samples:
    print(f"  ID range: {shtemaran_samples[0]['id']} to {shtemaran_samples[-1]['id']}")
    print(f"  Sample: id={shtemaran_samples[0]['id']}, "
          f"text_idx={shtemaran_samples[0]['text_index']}")
    print(f"    NP:   {shtemaran_samples[0]['original'][:80]}...")
    print(f"    Corr: {shtemaran_samples[0]['linguist_corrected'][:80]}...")

# Saving as JSONL (input for gemini_shtemaran_runner.py)
jsonl_path = Path("./Data Full/shtemaran_292_benchmark.jsonl")
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for s in shtemaran_samples:
        rec = {"id": s["id"], "text_index": s["text_index"],
               "original": s["original"],
               "linguist_corrected": s["linguist_corrected"]}
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f"\nSaved to {jsonl_path}")
print(f">>> Next: run  python gemini_shtemaran_runner.py --api-key YOUR_KEY")

# Label distribution
label_counts = Counter()
for s in shtemaran_samples: label_counts.update(s["labels"])
total_tokens = sum(label_counts.values())
print(f"\nLabel distribution ({total_tokens:,} tokens):")
for i, name in enumerate(LABEL_LIST):
    c = label_counts.get(i, 0)
    print(f"  {name:15s}: {c:>6,} ({100*c/total_tokens:.2f}%)")
needs_change = sum(1 for s in shtemaran_samples if any(l != 0 for l in s['labels']))
print(f"Sentences needing changes: {needs_change}/{len(shtemaran_samples)}")

Partial_NP_Final.txt: 292 lines
ONLY_Participle-related: 292 lines

Shtemaran dataset: 292 sentence pairs
  Skipped: 0
  ID range: 1 to 292
  Sample: id=1, text_idx=1
    NP:   – Ինչ ես խորհում – սթափվելով մտքերից դարձավ նա Վարդանին – լինելիքը կլինի:...
    Corr: – Ինչ ես խորհում – սթափվելով մտքերից՝ դարձավ նա Վարդանին – լինելիքը կլինի:...

Saved to Data Full/shtemaran_292_benchmark.jsonl
>>> Next: run  python gemini_shtemaran_runner.py --api-key YOUR_KEY

Label distribution (4,827 tokens):
  O              :  4,589 (95.07%)
  COMMA_AFTER    :    100 (2.07%)
  BUTH_AFTER     :    138 (2.86%)
  REMOVE_COMMA   :      0 (0.00%)
Sentences needing changes: 167/292


## Step 4: Load Neural Models

In [189]:
class BiLSTMPunctuator(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers,
                 dropout, num_classes, pretrained_emb=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if pretrained_emb is not None:
            self.embedding.weight.data.copy_(pretrained_emb)
            self.embedding.weight.requires_grad = False
        self.lstm = nn.LSTM(input_size=emb_dim, hidden_size=hidden_size,
                           num_layers=num_layers, batch_first=True,
                           bidirectional=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)
    def forward(self, x):
        out, _ = self.lstm(self.embedding(x))
        return self.classifier(self.dropout(out))

with open(BILSTM_VOCAB, 'r', encoding='utf-8') as f:
    bilstm_vocab = json.load(f)
ckpt = torch.load(BILSTM_CKPT, map_location="cpu", weights_only=False)
bp = ckpt.get("best_params", ckpt.get("hyperparameters", {}))
bilstm_model = BiLSTMPunctuator(
    vocab_size=ckpt["vocab_size"], emb_dim=ckpt["embedding_dim"],
    hidden_size=bp["hidden_size"], num_layers=bp["num_layers"],
    dropout=bp["dropout"], num_classes=ckpt["num_classes"])
bilstm_model.load_state_dict(ckpt["model_state_dict"])
bilstm_model.eval()
print(f"BiLSTM loaded: hidden={bp['hidden_size']}, layers={bp['num_layers']}")

BiLSTM v3 loaded: hidden=128, layers=1


In [190]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

mbert_tokenizer = AutoTokenizer.from_pretrained(MBERT_NAME)
mbert_model = AutoModelForTokenClassification.from_pretrained(MBERT_NAME, num_labels=NUM_LABELS)
mbert_model.load_state_dict(torch.load(MBERT_WEIGHTS, map_location="cpu", weights_only=True))
mbert_model.eval()
print(f"mBERT loaded: {sum(p.numel() for p in mbert_model.parameters()):,} params")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

mBERT loaded: 177,265,924 params


In [192]:
hyebert_model, hyebert_tokenizer = None, None
if HYEBERT_AVAILABLE:
    try:
        hyebert_tokenizer = AutoTokenizer.from_pretrained(HYEBERT_NAME)
        hyebert_model = AutoModelForTokenClassification.from_pretrained(
            HYEBERT_NAME, num_labels=NUM_LABELS)
        hyebert_model.load_state_dict(
            torch.load(HYEBERT_WEIGHTS, map_location="cpu", weights_only=True))
        hyebert_model.eval()
        print(f"HyeBERT loaded: {sum(p.numel() for p in hyebert_model.parameters()):,} params")
    except Exception as e:
        print(f"HyeBERT failed: {e}"); HYEBERT_AVAILABLE = False
else:
    print("HyeBERT weights not found -- skipping")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

HyeBERT v3 loaded: 65,967,364 params


## Step 5: Neural Model Inference

In [196]:
UNK_IDX = bilstm_vocab.get("<UNK>", 1)

def get_bilstm_probs(words):
    ids = [bilstm_vocab.get(w, UNK_IDX) for w in words[:256]]
    x = torch.tensor([ids], dtype=torch.long)
    with torch.no_grad(): logits = bilstm_model(x)
    return F.softmax(logits[0], dim=-1).numpy()

def get_bert_probs(words, tokenizer, model):
    enc = tokenizer(words, is_split_into_words=True,
                    max_length=128, truncation=True, padding=False, return_tensors="pt")
    wids = enc.word_ids(batch_index=0)
    with torch.no_grad():
        logits = model(input_ids=enc["input_ids"],
                       attention_mask=enc["attention_mask"]).logits[0]
    probs = F.softmax(logits, dim=-1).numpy()
    word_probs, prev = [], None
    for i, wid in enumerate(wids):
        if wid is None: continue
        if wid != prev: word_probs.append(probs[i])
        prev = wid
    return np.array(word_probs)

print("Prediction functions ready \u2713")

Prediction functions ready ✓


In [198]:
print("Running neural inference on Shtemaran benchmark...")
t0 = time.time()
for s in shtemaran_samples:
    words, labels = s["words"], s["labels"]
    bp_arr = get_bilstm_probs(words)
    mp = get_bert_probs(words, mbert_tokenizer, mbert_model)
    hp = get_bert_probs(words, hyebert_tokenizer, hyebert_model) if HYEBERT_AVAILABLE else None

    n = min(len(labels), len(bp_arr), len(mp))
    if hp is not None: n = min(n, len(hp))
    s["n_aligned"] = n
    s["labels_aligned"] = labels[:n]
    s["bilstm_preds"] = bp_arr[:n].argmax(axis=1).tolist()
    s["mbert_preds"] = mp[:n].argmax(axis=1).tolist()
    if hp is not None: s["hyebert_preds"] = hp[:n].argmax(axis=1).tolist()
    combined = ENSEMBLE_ALPHA * bp_arr[:n] + (1 - ENSEMBLE_ALPHA) * mp[:n]
    s["ensemble_preds"] = combined.argmax(axis=1).tolist()

print(f"Done in {time.time()-t0:.1f}s ({len(shtemaran_samples)} sentences)")

Running neural inference on Shtemaran benchmark...
Done in 22.8s (292 sentences)


## Step 6: Load Gemini Cache

Run `python gemini_shtemaran_runner.py --api-key YOUR_KEY` first.

In [201]:
gemini_available = False
if GEMINI_CACHE.exists():
    with open(GEMINI_CACHE, 'r', encoding='utf-8') as f:
        gemini_results = json.load(f)
    gemini_available = True
    print(f"Gemini cache loaded: {len(gemini_results)} results")

    for s in shtemaran_samples:
        gem_text = gemini_results.get(str(s["id"]), s["original"])
        orig_tokens = tokenize_armenian(s["original"])
        gem_tokens = tokenize_armenian(gem_text)
        gem_labels = generate_labels(orig_tokens, gem_tokens)
        _, gem_word_labels = extract_word_labels(orig_tokens, gem_labels)
        n = s["n_aligned"]
        s["gemini_preds"] = (gem_word_labels[:n] + [0] * n)[:n]
        s["gemini_output"] = gem_text
    print("Gemini labels generated \u2713")

    gem_changed = sum(1 for s in shtemaran_samples
                      if any(p != 0 for p in s["gemini_preds"]))
    print(f"Gemini made changes in {gem_changed}/{len(shtemaran_samples)} sentences")
else:
    print("\u26a0 Gemini cache not found!")
    print(f"  Expected: {GEMINI_CACHE}")
    print(f"  Run: python gemini_shtemaran_runner.py --api-key YOUR_KEY")

Gemini cache loaded: 292 results
Gemini labels generated ✓
Gemini made changes in 207/292 sentences


## Step 7: Evaluation Functions

In [204]:
def evaluate_model(samples, pred_key, model_name, verbose=True):
    all_preds, all_labels = [], []
    for s in samples:
        if pred_key not in s: continue
        all_preds.extend(s[pred_key])
        all_labels.extend(s["labels_aligned"])
    if not all_preds:
        if verbose: print(f"\n{model_name}: No predictions")
        return None
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    ins_f1 = f1_score(all_labels, all_preds, labels=[1, 2], average="macro", zero_division=0)
    preds_3c = [0 if x == 3 else x for x in all_preds]
    labels_3c = [0 if x == 3 else x for x in all_labels]
    f1_3c = f1_score(labels_3c, preds_3c, labels=[0,1,2], average="macro", zero_division=0)
    p_per, r_per, f1_per, _ = precision_recall_fscore_support(
        all_labels, all_preds, labels=list(range(NUM_LABELS)), zero_division=0)
    if verbose:
        print(f"\n{'='*65}")
        print(f"{model_name} -- Shtemaran Benchmark")
        print(f"{'='*65}")
        # Pass labels= explicitly to handle cases where not all classes appear
        print(classification_report(all_labels, all_preds,
              labels=list(range(NUM_LABELS)),
              target_names=LABEL_LIST, digits=3, zero_division=0))
        pc, lc = Counter(all_preds), Counter(all_labels)
        print("Distribution:")
        for i, name in enumerate(LABEL_LIST):
            print(f"  {name:15s}: {lc.get(i,0):>6,} true, {pc.get(i,0):>6,} pred")
        print(f"\nMacro-F1:          {macro_f1:.4f}")
        print(f"3-class remap F1:  {f1_3c:.4f}")
        print(f"Insertion-only F1: {ins_f1:.4f}")
    return {"model": model_name,
            "macro_f1": round(macro_f1, 5), "f1_3c": round(f1_3c, 5), "ins_f1": round(ins_f1, 5),
        "per_class_p": {LABEL_LIST[i]: round(float(p_per[i]), 4) for i in range(NUM_LABELS)},
        "per_class_r": {LABEL_LIST[i]: round(float(r_per[i]), 4) for i in range(NUM_LABELS)},
        "per_class_f1": {LABEL_LIST[i]: round(float(f1_per[i]), 4) for i in range(NUM_LABELS)},
        "all_preds": all_preds, "all_labels": all_labels}

def correction_coverage(samples, pred_key, model_name, verbose=True):
    total_gold, exact, false_ins = 0, 0, 0
    class_gold, class_matched = Counter(), Counter()
    for s in samples:
        if pred_key not in s: continue
        for lbl, pred in zip(s["labels_aligned"], s[pred_key]):
            if lbl != 0:
                total_gold += 1; class_gold[lbl] += 1
                if pred == lbl: exact += 1; class_matched[lbl] += 1
            elif pred != 0: false_ins += 1
    if total_gold == 0: return None
    coverage = exact / total_gold
    precision = exact / (exact + false_ins) if (exact + false_ins) > 0 else 0
    if verbose:
        print(f"\n{'='*65}")
        print(f"{model_name} -- Correction Coverage")
        print(f"{'='*65}")
        print(f"  Linguist corrections: {total_gold}")
        print(f"  Model matched:        {exact}")
        print(f"  False insertions:     {false_ins}")
        print(f"  Coverage: {exact}/{total_gold} = {coverage:.1%}")
        print(f"  Precision: {exact}/{exact+false_ins} = {precision:.1%}")
        print(f"  Per-class:")
        for cls_id in sorted(class_gold.keys()):
            g, m = class_gold[cls_id], class_matched[cls_id]
            print(f"    {LABEL_LIST[cls_id]:15s}: {m:>4}/{g:>4} = {m/g:.1%}")
    return {"total_gold": total_gold, "exact": exact, "false_ins": false_ins,
            "coverage": round(coverage, 4), "precision": round(precision, 4)}

def sentence_accuracy(samples, pred_key, model_name):
    total = sum(1 for s in samples if pred_key in s)
    perfect = sum(1 for s in samples if pred_key in s and s[pred_key] == s["labels_aligned"])
    acc = perfect / total if total > 0 else 0
    print(f"  {model_name:30s}: {perfect}/{total} = {acc:.1%}")
    return {"perfect": perfect, "total": total, "accuracy": round(acc, 4)}

print("Evaluation functions ready \u2713")

Evaluation functions ready ✓


## Step 8: Run All Evaluations

In [207]:
print("=" * 70)
print("SHTEMARAN BENCHMARK RESULTS")
print("=" * 70)
bilstm_res = evaluate_model(shtemaran_samples, "bilstm_preds", "BiLSTM")
mbert_res = evaluate_model(shtemaran_samples, "mbert_preds", "mBERT")
ensemble_res = evaluate_model(shtemaran_samples, "ensemble_preds", f"Ensemble (a={ENSEMBLE_ALPHA})")
hyebert_res = None
if HYEBERT_AVAILABLE:
    hyebert_res = evaluate_model(shtemaran_samples, "hyebert_preds", "HyeBERT")
gemini_res = None
if gemini_available:
    gemini_res = evaluate_model(shtemaran_samples, "gemini_preds", "Gemini 2.5 Flash (0-shot CoT)")

SHTEMARAN BENCHMARK RESULTS

BiLSTM v3 -- Shtemaran Benchmark
              precision    recall  f1-score   support

           O      0.961     0.983     0.972      4587
 COMMA_AFTER      0.452     0.190     0.268       100
  BUTH_AFTER      0.294     0.182     0.225       137
REMOVE_COMMA      0.000     0.000     0.000         0

    accuracy                          0.944      4824
   macro avg      0.427     0.339     0.366      4824
weighted avg      0.931     0.944     0.936      4824

Distribution:
  O              :  4,587 true,  4,693 pred
  COMMA_AFTER    :    100 true,     42 pred
  BUTH_AFTER     :    137 true,     85 pred
  REMOVE_COMMA   :      0 true,      4 pred

Macro-F1:          0.3661
3-class remap F1:  0.4882
Insertion-only F1: 0.2464

mBERT -- Shtemaran Benchmark
              precision    recall  f1-score   support

           O      0.978     0.982     0.980      4587
 COMMA_AFTER      0.570     0.490     0.527       100
  BUTH_AFTER      0.585     0.555     0.5

## Step 9: Cross-Model Comparison

In [210]:
all_results = [
    ("BiLSTM", bilstm_res, "0.4104"),
    ("mBERT", mbert_res, "0.4854"),
    ("Ensemble (a=0.45)", ensemble_res, "0.4919"),
]
if hyebert_res: all_results.insert(1, ("HyeBERT", hyebert_res, "0.4110"))
if gemini_res: all_results.append(("Gemini Flash (0-shot)", gemini_res, "n/a"))

print("=" * 90)
print("CROSS-MODEL COMPARISON -- Shtemaran Benchmark")
print("=" * 90)
print(f"{'Model':<35} {'Macro-F1':>10} {'Ins F1':>8} {'3c F1':>8} {'Val F1':>8}")
print("-" * 75)
for name, res, val_f1 in all_results:
    if res:
        print(f"{name:<35} {res['macro_f1']:>10.4f} {res['ins_f1']:>8.4f} "
              f"{res['f1_3c']:>8.4f} {val_f1:>8}")

CROSS-MODEL COMPARISON -- Shtemaran Benchmark
Model                                 Macro-F1   Ins F1    3c F1   Val F1
---------------------------------------------------------------------------
BiLSTM v3                               0.3661   0.2464   0.4882   0.4104
HyeBERT v3                              0.3260   0.1688   0.4348   0.4110
mBERT                                   0.5190   0.5481   0.6920   0.4854
Ensemble (a=0.45)                       0.6745   0.5216   0.6745   0.4919
Gemini Flash (0-shot)                   0.6997   0.5600   0.6997      n/a


## Step 10: Correction Coverage & Sentence Accuracy

In [213]:
print("\n" + "=" * 70)
print("CORRECTION COVERAGE")
print("=" * 70)
cov_results = {}
cov_results["BiLSTM"] = correction_coverage(shtemaran_samples, "bilstm_preds", "BiLSTM")
cov_results["mBERT"] = correction_coverage(shtemaran_samples, "mbert_preds", "mBERT")
cov_results["Ensemble"] = correction_coverage(shtemaran_samples, "ensemble_preds", "Ensemble")
if HYEBERT_AVAILABLE:
    cov_results["HyeBERT"] = correction_coverage(shtemaran_samples, "hyebert_preds", "HyeBERT")
if gemini_available:
    cov_results["Gemini Flash"] = correction_coverage(shtemaran_samples, "gemini_preds", "Gemini Flash (0-shot)")

print("\nSentence-level accuracy:")
print("-" * 60)
for pk, nm in [("bilstm_preds","BiLSTM"),("mbert_preds","mBERT"),
               ("ensemble_preds","Ensemble"),("hyebert_preds","HyeBERT"),
               ("gemini_preds","Gemini Flash (0-shot)")]:
    if any(pk in s for s in shtemaran_samples):
        sentence_accuracy(shtemaran_samples, pk, nm)


CORRECTION COVERAGE

BiLSTM v3 -- Correction Coverage
  Linguist corrections: 237
  Model matched:        44
  False insertions:     79
  Coverage: 44/237 = 18.6%
  Precision: 44/123 = 35.8%
  Per-class:
    COMMA_AFTER    :   19/ 100 = 19.0%
    BUTH_AFTER     :   25/ 137 = 18.2%

mBERT -- Correction Coverage
  Linguist corrections: 237
  Model matched:        125
  False insertions:     83
  Coverage: 125/237 = 52.7%
  Precision: 125/208 = 60.1%
  Per-class:
    COMMA_AFTER    :   49/ 100 = 49.0%
    BUTH_AFTER     :   76/ 137 = 55.5%

Ensemble -- Correction Coverage
  Linguist corrections: 237
  Model matched:        109
  False insertions:     61
  Coverage: 109/237 = 46.0%
  Precision: 109/170 = 64.1%
  Per-class:
    COMMA_AFTER    :   42/ 100 = 42.0%
    BUTH_AFTER     :   67/ 137 = 48.9%

HyeBERT v3 -- Correction Coverage
  Linguist corrections: 237
  Model matched:        33
  False insertions:     114
  Coverage: 33/237 = 13.9%
  Precision: 33/147 = 22.4%
  Per-class:
    CO

## Step 11: Error Patterns

In [215]:
def show_confusion_detail(all_labels, all_preds, model_name):
    print(f"\n{model_name} -- Error patterns:")
    for tc in range(NUM_LABELS):
        for pc in range(NUM_LABELS):
            if tc == pc: continue
            count = sum(1 for l, p in zip(all_labels, all_preds) if l == tc and p == pc)
            if count > 0:
                print(f"  {LABEL_LIST[tc]:>15} -> {LABEL_LIST[pc]:<15}: {count:>5,}")

for nm, res in [("BiLSTM", bilstm_res), ("mBERT", mbert_res), ("Ensemble", ensemble_res)]:
    if res: show_confusion_detail(res["all_labels"], res["all_preds"], nm)
if gemini_res: show_confusion_detail(gemini_res["all_labels"], gemini_res["all_preds"], "Gemini Flash")


BiLSTM v3 -- Error patterns:
                O -> COMMA_AFTER    :    21
                O -> BUTH_AFTER     :    55
                O -> REMOVE_COMMA   :     3
      COMMA_AFTER -> O              :    75
      COMMA_AFTER -> BUTH_AFTER     :     5
      COMMA_AFTER -> REMOVE_COMMA   :     1
       BUTH_AFTER -> O              :   110
       BUTH_AFTER -> COMMA_AFTER    :     2

mBERT -- Error patterns:
                O -> COMMA_AFTER    :    33
                O -> BUTH_AFTER     :    49
                O -> REMOVE_COMMA   :     1
      COMMA_AFTER -> O              :    46
      COMMA_AFTER -> BUTH_AFTER     :     5
       BUTH_AFTER -> O              :    57
       BUTH_AFTER -> COMMA_AFTER    :     4

Ensemble -- Error patterns:
                O -> COMMA_AFTER    :    26
                O -> BUTH_AFTER     :    35
      COMMA_AFTER -> O              :    53
      COMMA_AFTER -> BUTH_AFTER     :     5
       BUTH_AFTER -> O              :    68
       BUTH_AFTER -> COMMA_AFTER   

## Step 12: Visualizations

In [218]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plot_data = [("BiLSTM", bilstm_res, cov_results.get("BiLSTM")),
             ("mBERT", mbert_res, cov_results.get("mBERT")),
             ("Ensemble", ensemble_res, cov_results.get("Ensemble"))]
if hyebert_res: plot_data.insert(1, ("HyeBERT", hyebert_res, cov_results.get("HyeBERT")))
if gemini_res: plot_data.append(("Gemini Flash", gemini_res, cov_results.get("Gemini Flash")))

names = [n for n, r, _ in plot_data if r]
macro_f1s = [r["macro_f1"] for _, r, _ in plot_data if r]
ins_f1s = [r["ins_f1"] for _, r, _ in plot_data if r]
covs = [c["coverage"] if c else 0 for _, r, c in plot_data if r]
colors = ["#4C72B0","#55A868","#C44E52","#8172B2","#CCB974"][:len(names)]

for ax, vals, title, fmt in [
    (axes[0], macro_f1s, "Macro-F1", ".4f"),
    (axes[1], ins_f1s, "Insertion-Only F1", ".4f"),
    (axes[2], [c*100 for c in covs], "Correction Coverage (%)", ".1f")]:
    bars = ax.bar(names, vals, color=colors, edgecolor="white", linewidth=1.5)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.3 if vals else 1)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
                f"{v:{fmt}}", ha="center", fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.3)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.suptitle("Shtemaran Benchmark -- Model Comparison", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("shtemaran_benchmark_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: shtemaran_benchmark_comparison.png")

Saved: shtemaran_benchmark_comparison.png


/var/folders/7c/pvybjh094kv1kx3g5xlrqlg80000gp/T/ipykernel_68744/3076544206.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [220]:
fig, ax = plt.subplots(figsize=(12, 6))
classes = ["COMMA_AFTER", "BUTH_AFTER", "REMOVE_COMMA"]
x = np.arange(len(classes))
width = 0.15
for i, (nm, res, _) in enumerate(plot_data):
    if res:
        vals = [res["per_class_f1"].get(c, 0) for c in classes]
        ax.bar(x + (i - len(plot_data)/2) * width, vals, width*0.9,
               label=nm, color=colors[i], edgecolor="white")
ax.set_title("Per-Class F1 -- Shtemaran", fontsize=14, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(classes)
ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("shtemaran_per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/7c/pvybjh094kv1kx3g5xlrqlg80000gp/T/ipykernel_68744/3698347934.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [222]:
gold2k = {"BiLSTM": 0.4116, "mBERT": 0.4655, "Ensemble": 0.4648}
stm = {n: r["macro_f1"] for n, r, _ in plot_data if r and n in gold2k}
common = [m for m in gold2k if m in stm]
if common:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(common))
    g2k = [gold2k[m] for m in common]; st = [stm[m] for m in common]
    ax.bar(x-0.15, g2k, 0.3, label="Gold 2K", color="#4C72B0")
    ax.bar(x+0.15, st, 0.3, label="Shtemaran", color="#C44E52")
    for i, m in enumerate(common):
        d = st[i]-g2k[i]; sign = "+" if d > 0 else ""
        ax.annotate(f"d={sign}{d:.4f}", (x[i], max(g2k[i],st[i])+0.005),
                   ha="center", fontsize=9, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(common); ax.set_ylabel("Macro-F1")
    ax.set_title("Gold 2K vs Shtemaran -- Macro-F1", fontsize=14, fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig("shtemaran_vs_gold2k.png", dpi=150, bbox_inches="tight")
    plt.show()

/var/folders/7c/pvybjh094kv1kx3g5xlrqlg80000gp/T/ipykernel_68744/2211771583.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [224]:
fig, ax = plt.subplots(figsize=(10, 8))
markers = ["o", "s", "^", "D", "P"]
for i, (nm, res, _) in enumerate(plot_data):
    if not res: continue
    for cls in ["COMMA_AFTER", "BUTH_AFTER"]:
        p = res["per_class_p"].get(cls, 0)
        r = res["per_class_r"].get(cls, 0)
        ax.scatter(r, p, s=120, marker=markers[i], color=colors[i],
                  edgecolors="black", linewidth=0.5, zorder=5)
        ax.annotate(f"{nm}\n{cls[:5]}", (r, p), textcoords="offset points",
                   xytext=(8, 8), fontsize=7)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision vs Recall -- Minority Classes", fontsize=14, fontweight="bold")
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.plot([0,1],[0,1],"k--",alpha=0.2); ax.grid(alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig("shtemaran_precision_recall.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/7c/pvybjh094kv1kx3g5xlrqlg80000gp/T/ipykernel_68744/2632130141.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 13: Qualitative Error Analysis

In [227]:
print("=" * 80)
print("SAMPLE PREDICTIONS -- sentences with disagreements")
print("=" * 80)
n_shown = 0
for s in shtemaran_samples:
    if n_shown >= 15: break
    if all(l == 0 for l in s["labels_aligned"]): continue
    pred_keys = ["bilstm_preds","mbert_preds","ensemble_preds"]
    if gemini_available: pred_keys.append("gemini_preds")
    preds_set = set(tuple(s.get(pk, [])) for pk in pred_keys if pk in s)
    if len(preds_set) <= 1: continue
    print(f"\n--- Sentence {s['id']} ---")
    print(f"  Original:  {s['original']}")
    print(f"  Gold:      {s['linguist_corrected']}")
    for pk, nm in [("bilstm_preds","BiLSTM"),("mbert_preds","mBERT"),
                   ("ensemble_preds","Ensemble"),("gemini_preds","Gemini")]:
        if pk in s:
            match = "Y" if s[pk] == s["labels_aligned"] else "N"
            recon = reconstruct_sentence(s["words"][:s["n_aligned"]], s[pk])
            print(f"  {nm:10s}: {recon}  [{match}]")
    n_shown += 1

SAMPLE PREDICTIONS -- sentences with disagreements

--- Sentence 2 ---
  Original:  Վարդանը հայացքը հատակին խոնարհած մտմտում էր ասելիքը:
  Gold:      Վարդանը, հայացքը հատակին խոնարհած, մտմտում էր ասելիքը:
  BiLSTM    : Վարդանը , հայացքը հատակին խոնարհած մտմտում էր ասելիքը  [N]
  mBERT     : Վարդանը , հայացքը հատակին խոնարհած , մտմտում էր ասելիքը  [Y]
  Ensemble  : Վարդանը , հայացքը հատակին խոնարհած , մտմտում էր ասելիքը  [Y]
  Gemini    : Վարդանը , հայացքը հատակին խոնարհած , մտմտում էր ասելիքը  [Y]

--- Sentence 3 ---
  Original:  Երբ արդեն թողել էինք Սոֆիան և մեր ինքնաթիռը թռչնի թեթևությամբ սահում էր ինը հազար մետրով չափվող բարձունքներում ուղեկցորդուհին մեզ պարզեց պատվոգրի մեծությամբ մի թուղթ վրան երկրագնդի պատկերը բոլորված ոսկեզօծ նախշաժապավենով:
  Gold:      Երբ արդեն թողել էինք Սոֆիան և մեր ինքնաթիռը թռչնի թեթևությամբ սահում էր ինը հազար մետրով չափվող բարձունքներում ուղեկցորդուհին մեզ պարզեց պատվոգրի մեծությամբ մի թուղթ վրան երկրագնդի պատկերը՝ բոլորված ոսկեզօծ նախշաժապավենով:
  BiLS

## Step 14: McNemar's Test

In [230]:
from scipy.stats import chi2

def mcnemar_test(labels, preds_a, preds_b, model_a, model_b):
    a_right_b_wrong = sum(1 for l,a,b in zip(labels,preds_a,preds_b) if a==l and b!=l)
    a_wrong_b_right = sum(1 for l,a,b in zip(labels,preds_a,preds_b) if a!=l and b==l)
    n = a_right_b_wrong + a_wrong_b_right
    if n == 0:
        print(f"  {model_a} vs {model_b}: No discordant pairs"); return
    stat = (abs(a_right_b_wrong - a_wrong_b_right) - 1)**2 / n
    p_val = 1 - chi2.cdf(stat, df=1)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "n.s."
    print(f"  {model_a} vs {model_b}:")
    print(f"    {model_a} right, {model_b} wrong: {a_right_b_wrong}")
    print(f"    {model_a} wrong, {model_b} right: {a_wrong_b_right}")
    print(f"    McNemar X2={stat:.2f}, p={p_val:.4f} {sig}")

print("McNemar's Test -- Pairwise (Shtemaran)")
print("-" * 65)
pairs = [(bilstm_res, mbert_res, "BiLSTM", "mBERT"),
         (mbert_res, ensemble_res, "mBERT", "Ensemble")]
if gemini_res:
    pairs.append((ensemble_res, gemini_res, "Ensemble", "Gemini"))
    pairs.append((bilstm_res, gemini_res, "BiLSTM", "Gemini"))
for ra, rb, na, nb in pairs:
    if ra and rb:
        mcnemar_test(ra["all_labels"], ra["all_preds"], rb["all_preds"], na, nb)

McNemar's Test -- Pairwise (Shtemaran)
-----------------------------------------------------------------
  BiLSTM vs mBERT:
    BiLSTM right, mBERT wrong: 74
    BiLSTM wrong, mBERT right: 151
    McNemar X2=25.67, p=0.0000 ***
  mBERT vs Ensemble:
    mBERT right, Ensemble wrong: 20
    mBERT wrong, Ensemble right: 26
    McNemar X2=0.54, p=0.4610 n.s.
  Ensemble vs Gemini:
    Ensemble right, Gemini wrong: 126
    Ensemble wrong, Gemini right: 101
    McNemar X2=2.54, p=0.1112 n.s.
  BiLSTM vs Gemini:
    BiLSTM right, Gemini wrong: 132
    BiLSTM wrong, Gemini right: 190
    McNemar X2=10.09, p=0.0015 **


## Step 15: Save Results

In [232]:
results_summary = {
    "evaluation_set": "shtemaran_292",
    "n_sentences": len(shtemaran_samples),
    "n_tokens": sum(s["n_aligned"] for s in shtemaran_samples),
    "ensemble_alpha": ENSEMBLE_ALPHA,
}
for key, res, cov_key in [("bilstm", bilstm_res, "BiLSTM"),
                           ("mbert", mbert_res, "mBERT"),
                           ("ensemble", ensemble_res, "Ensemble")]:
    if res:
        results_summary[key] = {k:v for k,v in res.items() if k not in ("all_preds","all_labels")}
        cov = cov_results.get(cov_key)
        if cov: results_summary[key]["coverage"] = cov
if hyebert_res:
    results_summary["hyebert"] = {k:v for k,v in hyebert_res.items() if k not in ("all_preds","all_labels")}
if gemini_res:
    results_summary["gemini_flash_zeroshot"] = {k:v for k,v in gemini_res.items() if k not in ("all_preds","all_labels")}
    cov = cov_results.get("Gemini Flash")
    if cov: results_summary["gemini_flash_zeroshot"]["coverage"] = cov
results_summary["gold_2k_reference"] = {"bilstm": 0.4116, "mbert": 0.4655, "ensemble": 0.4648}
results_summary["val_reference"] = {"bilstm": 0.4104, "mbert": 0.4854, "ensemble": 0.4919}

with open("shtemaran_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)
print("Results saved to shtemaran_benchmark_results.json")

Results saved to shtemaran_benchmark_results.json


>This notebook evaluates all models on the Shtemaran 292 benchmark and produces
Tables II, Figures 1-2, and McNemar's test results reported in the paper.